AIM MIGRATION ( 30-12-2021)

1. First backup the data files from AIM production server to local machine
    -   scp -P 8288 root@sg02.synercatalyst.com://var/lib/perfectwork/SG02/CONTAINERS/SG02_AIM_SG02DB/backups/2022_01_02_17_40_27_aim.zip .

2. Start the PW.2.0 services
    -   cd /opt/PW/PW.2.0
    -   ./odoo-bin -c pw_sg01.conf

3. Restore the database --> PW2-AIM

4. Remove the following modules from the PW2-AIM database
    - project_timeline_critical_path
    - backend_theme_v11
    - theme_art
    - digest
    - account_accountant_pw
    - mass_mailing_resend
    - sale_coupon

5. Conduct full database upgrade using PW.2.0
    - ./odoo-bin -c pw_sg01.conf -u all

6. Backup database PW2-AIM

7. Restore PW2-AIM to PWTO2.5

8.  Migrating database using OpenUpgrade_12.0
   - cd /opt/PW/OpenUpgrade_12.0
   - ./odoo-bin -c pw.conf --stop-after-init  -u all -d PWTO2.5

11. Conduct quick test run using PW.2.5
   - cd /opt/PW/PW.2.5
   - ./odoo-bin -c pw.conf
    
12. Conduct database update for all module - data consistancy check
    - ./odoo-bin -c pw.conf -u all -d PWTO2.5

    _A lot of warning and errors as many modules are not available for loading_
    ERROR PWTO2.5 odoo.modules.loading: Some modules have inconsistent states, some dependencies may be missing: ['hr_skill', 'mass_editing', 'mass_mailing_resend', 'perfectwork_sg_aim', 'sale_coupon', 'website_sale_coupon'] 

13. Backup database PWTO2.5

14. Restore Database PWTO3.0

15. Migrating database using OpenUpgrade_13.0
   - cd /opt/PW/OpenUpgrade_13.0
   - ./odoo-bin -c pw.conf --stop-after-init  -u all -d PWTO3.0

15. Conduct quick test run using PW.3.0
   - cd /opt/PW/PW.3.0
   - ./odoo-bin -c pw.conf

    
16. Conduct database update for all module - data consistancy check
    - ./odoo-bin -c pw.conf -u all -d PWTO3.0

17. Uninstall the following modules
    - website_event

18. Install the following modules
    - pw_theme_layout
    - website_event_sale

19. Backup PWTO3.0 database
   
20. Shutdown the AIM-PW2 Container

21. Prepare the docker container for AIM PW3 using PW_CS
    
22. Restore PWTO3.0 database to new AIM-PW3 Container

23. Change system parameters
    - web.base.url --> asiainstituteofmentoring.com
    - report.url --> asiainstituteofmentoring.com

24. Configure website settings
    - Install a theme
    - Save the website settings.
    - Update all modules

25. Need to clean up unwanted view for Survey ( Maybe other module as well )
    - Settings - View: Remove survey_init and survey.page
    - Remove some of the orphan pages

26. Run user script below for portal user permission
    
27. Testing ...



In [2]:
from odooly import Client, Env
from odoo import api, fields, models, _

source_connection = Client.from_config('aim')
print (source_connection)


<Client 'http://localhost:8069/xmlrpc#PWTO3.0'>


In [ ]:
from odooly import Client, Env
from odoo import api, fields, models, _


source_connection = Client(server='http://localhost:8069', db='PWTO3.0', user='admin.synercatalyst', password='Wengseng1@')

res_user_obj = source_connection.env['res.users']
user_ids = res_user_obj.search([(1, '=', 1)])

internal_groups = [1, 10]

for user in user_ids:
    user_groups = user.groups_id
    for group in user_groups:
        if (group.id == 1):
            print(user.name, group.name)
            user.groups_id = [(3, 10)]
        if (group.id == 74):
            print(user.name, group.name)
            user.groups_id = [(3, 10)]

In [ ]:
# We may need to clear the mass mailing field - mailing_domain, so we can add new domain name inside
# If not, the mailing domain is always with [('list_ids', 'in', [])], which will not be overrite
# Then no new mailing list will be added into the form

from odooly import Client, Env
from odoo import api, fields, models, _


source_connection = Client(server='https://asiainstituteofmentoring.com', db='PW3-AIM', user='admin.synercatalyst', password='Wengseng1@')

mailing_mailing_obj = source_connection.env['mailing.mailing']
mailing_ids = mailing_mailing_obj.search([(1, '=', 1)])


for mailing in mailing_ids:
    if mailing.mailing_domain == "[('list_ids', 'in', [])]":
        mailing.mailing_domain = ""
        print (mailing.name)

In [1]:
# AIM - Clean Up mailing.contact 
# Each contact should have one entry in mailing.contact

from odooly import Client, Env
from odoo import api, fields, models, _


source_connection = Client(server='https://asiainstituteofmentoring.com', db='PW3-AIM', user='admin.synercatalyst', password='Wengseng1@')

res_partner_ids = source_connection.env['res.partner'].search([(1, '=', 1)])
mailing_subscription_obj = source_connection.env['mailing.contact.subscription']

# Need to start by using the res.partner
for partner in res_partner_ids:   
    # If there is an email, then we search mailing.contact
    if partner.email:
        # Scan thru all the mailing.list.contacts with the following email
        mailing_contact_ids = source_connection.env['mailing.contact'].search([('email', '=', partner.email)])
        # Only process the mailing.contact if > 1
        if len(mailing_contact_ids) > 1:
            i = 1
            for contact in mailing_contact_ids:
                if i == 1:
                    d_contact_id = contact.id
                    print('WORKING ON CONTACT : %s' % contact.name )
                    d_contact_ids = []
                    d_list_ids = []
                    d_opt_out_ids = []
                    i += 1
                else:
                    # Process additional mailing.contact with same contact name
                    # If the contact does not have mailing list - Remove the mailing list contact
                    contact_mailing_lists = contact.subscription_list_ids
                    if not contact_mailing_lists:
                        print('MAIN CONTACT %s - Empty Mailing List' % contact.name)
                        contact.unlink()
                    else:
                        for item in contact_mailing_lists:
                            # Store list_id and contact_id into array d_contact_ids[] and d_list_ids{}
                            print('CREATE %s INTO %s' % (item.list_id.name, contact.name))
                            # Due to duplicate mailing list, so we need to check the list before append
                            exist_count = d_list_ids.count(item.list_id.id)
                            if exist_count == 0:
                                d_list_ids.append(item.list_id.id)
                                d_contact_ids.append(item.contact_id.id)
                                d_opt_out_ids.append(item.opt_out)
                            item.unlink()
                        # Remove the more_contact in mailing.contact
                    contact.unlink()

            # Now we need to create new mailing.contact.subscription records inside contact
            # Search for the record and insert into the mailing list
            main_contact = source_connection.env['mailing.contact'].search([('id', '=', d_contact_id )])
            j = 0
            while j < len(d_list_ids):
                try:
                    main_contact.write({'subscription_list_ids': [(0,0,{'list_id':d_list_ids[j], 'contact_id':d_contact_ids[j], 'opt_out':d_opt_out_ids[j],})]})
                except:
                    print('SKIP RECORD DUE TO ERROR ....')
                j += 1

WORKING ON CONTACT : May
CREATE Releasing Workshop INTO May
WORKING ON CONTACT : Lyn Wong
CREATE AIM 21 Wave 1 Mailing List INTO Lyn Wong
WORKING ON CONTACT : TAN BAKCHAI 
CREATE AIM 21 Wave 2 Mailing List INTO TAN BAKCHAI 
CREATE AIM 21 Wave 3 Mailing List INTO TAN BAKCHAI 
WORKING ON CONTACT : Mey Thir
CREATE Releasing Workshop INTO Mey Thir
WORKING ON CONTACT : Vernessa Chuah
CREATE Releasing Workshop INTO Vernessa Chuah
WORKING ON CONTACT : Anupma Tiwary
CREATE AIM 21 Wave 1 Mailing List INTO Anupma Tiwary
CREATE AIM 21 Wave 3 Mailing List INTO Anupma Tiwary
WORKING ON CONTACT : Matthew Soon Min Hian
CREATE AIM Mailing List INTO Matthew Soon Min Hian
WORKING ON CONTACT : Mukund Vijayaraghavan
CREATE AIM 21 Wave 1 Mailing List INTO Mukund Vijayaraghavan
WORKING ON CONTACT : Franciane Belanger
CREATE AIM 21 Wave 1 Mailing List INTO Franciane Belanger
WORKING ON CONTACT : Jamie Tang
CREATE AIM 21 Wave 1 Mailing List INTO Jamie Tang
WORKING ON CONTACT : Judy Prince 
CREATE AIM Mailing 